This notebook is for data preprocessing check.


Dropped Time

Removed features with more than 50% missing values

Imputed remaining missing values with median

Removed constant features

Performed stratified 70/15/15 split

Standardized features using training statistics

vae


VAE data saved to ../data/preprocessed_data_outlier_vae.pkl

In [45]:
#import package
import pandas as pd
import numpy as np
import sys
import os
from sklearn.model_selection import train_test_split

sys.path.append(os.path.abspath(".."))
from src.utils import RANDOM_SEED, TRAIN_RATIO, VAL_RATIO, TEST_RATIO

In [46]:
print(sys.version)

3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]


In [47]:
df_raw = pd.read_csv("../data/uci-secom.csv")
df = df_raw.copy() #use copy to not change the origianl file

In [48]:
print("Shape:", df.shape)

Shape: (1567, 592)


In [49]:
#delete pass/fail for not data leak
# Remove Time from the input features.
# We use only sensor-related features for the first baseline experiment.
y = df["Pass/Fail"].copy()
X = df.drop(columns=["Pass/Fail"]).copy()

print("Before dropping Time:", X.shape)

X = X.drop(columns=["Time"]).copy()

print("After dropping Time:", X.shape)

Before dropping Time: (1567, 591)
After dropping Time: (1567, 590)


In [50]:
#drop the data that have missing ratio bigger than 50%
missing_ratio = X.isnull().mean()
cols_to_drop = missing_ratio[missing_ratio > 0.5].index
print("Number of columns to drop:", len(cols_to_drop))

Number of columns to drop: 28


In [51]:
X = X.drop(columns=cols_to_drop).copy()
print("Shape after dropping high-missing columns:", X.shape)

Shape after dropping high-missing columns: (1567, 562)


In [52]:
#check the missing value percentage for the data that remains
print("Total missing values in X:", X.isnull().sum().sum())
missing_ratio_after = X.isnull().mean().sort_values(ascending=False)
missing_ratio_after.head(10)

Total missing values in X: 11683


385    0.456286
247    0.456286
112    0.456286
519    0.456286
566    0.174218
567    0.174218
568    0.174218
569    0.174218
563    0.174218
562    0.174218
dtype: float64

In [53]:
#fill the data with median (for each column)
X = X.fillna(X.median())
print("Total missing values after median imputation:", X.isnull().sum().sum())

Total missing values after median imputation: 0


In [54]:
#find the col that have same numbers in it 
constant_cols = [col for col in X.columns if X[col].nunique() == 1]

print("Number of constant features:", len(constant_cols))
print("Constant features:", constant_cols[:10])

Number of constant features: 116
Constant features: ['5', '13', '42', '49', '52', '69', '97', '141', '149', '178']


In [55]:
X = X.drop(columns=constant_cols).copy()
print("Shape after dropping constant features:", X.shape)

Shape after dropping constant features: (1567, 446)


In [56]:
print(y.value_counts())
print(y.value_counts(normalize=True))

Pass/Fail
-1    1463
 1     104
Name: count, dtype: int64
Pass/Fail
-1    0.933631
 1    0.066369
Name: proportion, dtype: float64


In [57]:
corr_matrix = X.corr().abs()

upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

to_drop_corr = [col for col in upper.columns if any(upper[col] > 0.95)]

print("Highly correlated features to drop:", len(to_drop_corr))

X = X.drop(columns=to_drop_corr)

print("Shape after correlation filtering:", X.shape)

Highly correlated features to drop: 174
Shape after correlation filtering: (1567, 272)


In [58]:
print (RANDOM_SEED)

559


In [59]:
# First split: train vs temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=(VAL_RATIO + TEST_RATIO),
    random_state=RANDOM_SEED,
    stratify=y
)

# Second split: validation vs test from the temporary set
temp_test_ratio = TEST_RATIO / (VAL_RATIO + TEST_RATIO)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=temp_test_ratio,
    random_state=RANDOM_SEED,
    stratify=y_temp
)

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)

X_train shape: (1096, 272)
X_val shape: (235, 272)
X_test shape: (236, 272)


In [60]:
print("Train label distribution:")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True))

print("\nValidation label distribution:")
print(y_val.value_counts())
print(y_val.value_counts(normalize=True))

print("\nTest label distribution:")
print(y_test.value_counts())
print(y_test.value_counts(normalize=True))

Train label distribution:
Pass/Fail
-1    1023
 1      73
Name: count, dtype: int64
Pass/Fail
-1    0.933394
 1    0.066606
Name: proportion, dtype: float64

Validation label distribution:
Pass/Fail
-1    220
 1     15
Name: count, dtype: int64
Pass/Fail
-1    0.93617
 1    0.06383
Name: proportion, dtype: float64

Test label distribution:
Pass/Fail
-1    220
 1     16
Name: count, dtype: int64
Pass/Fail
-1    0.932203
 1    0.067797
Name: proportion, dtype: float64


In [61]:
# Compute variance from training set only
train_var = X_train.var()

threshold = 0.01
low_variance_cols = train_var[train_var < threshold].index.tolist()

print("Number of low variance features:", len(low_variance_cols))
print("Some low variance features:", low_variance_cols[:10])

# Drop the same columns from all splits
X_train = X_train.drop(columns=low_variance_cols).copy()
X_val = X_val.drop(columns=low_variance_cols).copy()
X_test = X_test.drop(columns=low_variance_cols).copy()

print("X_train shape after low variance filtering:", X_train.shape)
print("X_val shape after low variance filtering:", X_val.shape)
print("X_test shape after low variance filtering:", X_test.shape)

Number of low variance features: 97
Some low variance features: ['7', '8', '9', '10', '11', '17', '20', '30', '53', '54']
X_train shape after low variance filtering: (1096, 175)
X_val shape after low variance filtering: (235, 175)
X_test shape after low variance filtering: (236, 175)


In [62]:
# Make sure y_train is a pandas Series
y_train_series = pd.Series(y_train, index=X_train.index)

# Compute absolute correlation between each feature and y
feature_scores = {}

for col in X_train.columns:
    corr_value = X_train[col].corr(y_train_series)
    
    # Handle possible NaN
    if pd.isna(corr_value):
        corr_value = 0.0
        
    feature_scores[col] = abs(corr_value)

# Convert to DataFrame and sort
score_df = pd.DataFrame({
    "feature": list(feature_scores.keys()),
    "score": list(feature_scores.values())
}).sort_values(by="score", ascending=False)

# Select top 100 features
top_k = 100
selected_cols = score_df.head(top_k)["feature"].tolist()

print("Top 10 selected features:")
print(score_df.head(10))
print("\nNumber of selected features:", len(selected_cols))

# Apply the same selected columns to train/val/test
X_train = X_train[selected_cols].copy()
X_val = X_val[selected_cols].copy()
X_test = X_test[selected_cols].copy()

print("Shapes after top-100 correlation selection:")
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

Top 10 selected features:
    feature     score
155     510  0.170844
39       59  0.159619
43       63  0.126515
130     431  0.121391
133     434  0.116371
18       28  0.113416
129     430  0.111884
99      196  0.111673
100     197  0.111116
121     336  0.110520

Number of selected features: 100
Shapes after top-100 correlation selection:
X_train: (1096, 100)
X_val: (235, 100)
X_test: (236, 100)


In [63]:
# Compute mean and standard deviation from the training set only
train_mean = X_train.mean()
train_std = X_train.std()

# Avoid division by zero just in case
train_std = train_std.replace(0, 1)

# Standardize train, validation, and test sets
X_train_scaled = (X_train - train_mean) / train_std
X_val_scaled = (X_val - train_mean) / train_std
X_test_scaled = (X_test - train_mean) / train_std

print("X_train_scaled shape:", X_train_scaled.shape)
print("X_val_scaled shape:", X_val_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)

X_train_scaled shape: (1096, 100)
X_val_scaled shape: (235, 100)
X_test_scaled shape: (236, 100)


In [64]:
print("Mean of first 5 scaled training features:")
print(X_train_scaled.iloc[:, :5].mean())

print("\nStd of first 5 scaled training features:")
print(X_train_scaled.iloc[:, :5].std())

Mean of first 5 scaled training features:
510   -1.166950e-16
59    -8.103818e-18
63    -5.915787e-17
431    1.296611e-17
434   -2.593222e-17
dtype: float64

Std of first 5 scaled training features:
510    1.0
59     1.0
63     1.0
431    1.0
434    1.0
dtype: float64


In [65]:
# ===== VAE latent feature extraction (put this after scaling cell) =====

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

# -----------------------------
# 1. Reproducibility and device
# -----------------------------
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# -----------------------------
# 2. Convert pandas DataFrame to numpy / tensor
# -----------------------------
X_train_np = X_train_scaled.values.astype(np.float32)
X_val_np = X_val_scaled.values.astype(np.float32)
X_test_np = X_test_scaled.values.astype(np.float32)

print("X_train_np shape:", X_train_np.shape)
print("X_val_np shape:", X_val_np.shape)
print("X_test_np shape:", X_test_np.shape)

train_tensor = torch.tensor(X_train_np, dtype=torch.float32)
val_tensor = torch.tensor(X_val_np, dtype=torch.float32)
test_tensor = torch.tensor(X_test_np, dtype=torch.float32)

batch_size = 64

train_loader = DataLoader(TensorDataset(train_tensor), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(val_tensor), batch_size=batch_size, shuffle=False)
test_loader = DataLoader(TensorDataset(test_tensor), batch_size=batch_size, shuffle=False)

# -----------------------------
# 3. Define VAE
# -----------------------------
input_dim = X_train_np.shape[1]
hidden_dim = 64
latent_dim = 20   # set to 20 to match your PCA comparison more directly

class VAE(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super().__init__()
        
        # Encoder
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
        
        # Decoder
        self.fc2 = nn.Linear(latent_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, input_dim)

    def encode(self, x):
        h = F.relu(self.fc1(x))
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + eps * std
        return z

    def decode(self, z):
        h = F.relu(self.fc2(z))
        x_recon = self.fc3(h)   # no sigmoid because inputs are standardized, not [0,1]
        return x_recon

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_recon = self.decode(z)
        return x_recon, mu, logvar

model = VAE(input_dim=input_dim, hidden_dim=hidden_dim, latent_dim=latent_dim).to(device)

# -----------------------------
# 4. Loss function
# -----------------------------
def vae_loss(x_recon, x, mu, logvar, beta=1.0):
    recon_loss = F.mse_loss(x_recon, x, reduction='sum')
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    total_loss = recon_loss + beta * kl_loss
    return total_loss, recon_loss, kl_loss

# -----------------------------
# 5. Training setup
# -----------------------------
lr = 1e-3
num_epochs = 150
beta = 1.0

optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# -----------------------------
# 6. Train VAE on training set only
# -----------------------------
for epoch in range(num_epochs):
    model.train()
    train_total = 0.0
    train_recon = 0.0
    train_kl = 0.0

    for batch in train_loader:
        x = batch[0].to(device)

        optimizer.zero_grad()
        x_recon, mu, logvar = model(x)
        loss, recon_loss, kl_loss = vae_loss(x_recon, x, mu, logvar, beta=beta)
        loss.backward()
        optimizer.step()

        train_total += loss.item()
        train_recon += recon_loss.item()
        train_kl += kl_loss.item()

    # Validation loss
    model.eval()
    val_total = 0.0
    val_recon = 0.0
    val_kl = 0.0

    with torch.no_grad():
        for batch in val_loader:
            x = batch[0].to(device)
            x_recon, mu, logvar = model(x)
            loss, recon_loss, kl_loss = vae_loss(x_recon, x, mu, logvar, beta=beta)

            val_total += loss.item()
            val_recon += recon_loss.item()
            val_kl += kl_loss.item()

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(
            f"Epoch [{epoch+1:3d}/{num_epochs}] | "
            f"Train Loss: {train_total/len(train_tensor):.4f} "
            f"(Recon: {train_recon/len(train_tensor):.4f}, KL: {train_kl/len(train_tensor):.4f}) | "
            f"Val Loss: {val_total/len(val_tensor):.4f} "
            f"(Recon: {val_recon/len(val_tensor):.4f}, KL: {val_kl/len(val_tensor):.4f})"
        )

# -----------------------------
# 7. Encode data into latent features
#    Use mu as deterministic latent representation
# -----------------------------
def encode_dataset(model, x_np, device):
    model.eval()
    x_tensor = torch.tensor(x_np, dtype=torch.float32).to(device)
    with torch.no_grad():
        mu, logvar = model.encode(x_tensor)
    return mu.cpu().numpy()

X_train_vae = encode_dataset(model, X_train_np, device)
X_val_vae = encode_dataset(model, X_val_np, device)
X_test_vae = encode_dataset(model, X_test_np, device)

print("X_train_vae shape:", X_train_vae.shape)
print("X_val_vae shape:", X_val_vae.shape)
print("X_test_vae shape:", X_test_vae.shape)

Using device: cuda
X_train_np shape: (1096, 100)
X_val_np shape: (235, 100)
X_test_np shape: (236, 100)
Epoch [  1/150] | Train Loss: 104.7165 (Recon: 103.9568, KL: 0.7597) | Val Loss: 116.5693 (Recon: 115.7270, KL: 0.8423)
Epoch [ 10/150] | Train Loss: 80.3203 (Recon: 74.3079, KL: 6.0125) | Val Loss: 86.9941 (Recon: 79.9245, KL: 7.0695)
Epoch [ 20/150] | Train Loss: 73.6779 (Recon: 65.4810, KL: 8.1969) | Val Loss: 81.3451 (Recon: 72.9116, KL: 8.4335)
Epoch [ 30/150] | Train Loss: 69.8438 (Recon: 60.2190, KL: 9.6248) | Val Loss: 79.3153 (Recon: 69.2013, KL: 10.1141)
Epoch [ 40/150] | Train Loss: 66.7918 (Recon: 56.3529, KL: 10.4389) | Val Loss: 76.1117 (Recon: 65.7178, KL: 10.3940)
Epoch [ 50/150] | Train Loss: 65.1721 (Recon: 54.2447, KL: 10.9274) | Val Loss: 74.1349 (Recon: 62.9510, KL: 11.1839)
Epoch [ 60/150] | Train Loss: 63.7251 (Recon: 52.2299, KL: 11.4951) | Val Loss: 73.6424 (Recon: 61.9391, KL: 11.7033)
Epoch [ 70/150] | Train Loss: 62.5566 (Recon: 50.5973, KL: 11.9594) | Val

In [66]:
import pickle
import pandas as pd

vae_data = {
    "X_train": pd.DataFrame(X_train_vae),
    "X_val": pd.DataFrame(X_val_vae),
    "X_test": pd.DataFrame(X_test_vae),
    "y_train": y_train,
    "y_val": y_val,
    "y_test": y_test,
    "note": "VAE latent feature version"
}

with open("../data/preprocessed_data_outlier_vae.pkl", "wb") as f:
    pickle.dump(vae_data, f)

print("VAE data saved to ../data/preprocessed_data_outlier_vae.pkl")

VAE data saved to ../data/preprocessed_data_outlier_vae.pkl
